# GAN: Generative Adversarial Nets (Goodfellow et al., NeurIPS 2014)
### Triển Khai Từ Con Số 0 (From Scratch) Bằng Python & NumPy

> **Tác giả:** Ian J. Goodfellow, Jean Pouget-Abadie, Mehdi Mirza, Bing Xu, David Warde-Farley, Sherjil Ozair, Aaron Courville, Yoshua Bengio  
> **Hội nghị:** NeurIPS / NIPS 2014 (Montreal, Canada)  
> **Ý nghĩa lịch sử:** Khai sinh khái niệm Mạng Đối Kháng Tạo Sinh (Generative Adversarial Networks), biến bài toán sinh dữ liệu thành một trò chơi Minimax hai người, tạo ra các hình ảnh sắc nét vượt trội so với các phương pháp cũ.

---

## 🎯 Mục Tiêu Bài Thực Hành
1. **Kiến trúc Discriminator & Generator:** Xây dựng hai mạng nơ-ron đối đầu độc lập bằng thuần Python và NumPy.
2. **Hàm mất mát Non-Saturating Loss:** Giải quyết hiện tượng triệt tiêu gradient của hàm Minimax gốc $\log(1 - D(G(z)))$ bằng cách tối đa hóa $\log D(G(z))$.
3. **Tối ưu hóa luân phiên (Alternating Optimization):** Huấn luyện $D$ phân biệt thật/giả và huấn luyện $G$ đánh lừa $D$.
4. **Academic Value Benchmark:** Đối chiếu định lượng sự khác biệt bản chất giữa Kỷ nguyên Tiền-GAN (Ước lượng mật độ MLE / Mode-Averaging) và GAN.

In [ ]:
import time
import numpy as np
print(f"[✓] Khởi tạo thư viện NumPy: {np.__version__}")

## PHẦN 1: CÁC KHỐI TẦNG NƠ-RON CƠ BẢN (FORWARD & BACKWARD)

Cài đặt các tầng tuyến tính `LinearLayer`, hàm kích hoạt `LeakyReLULayer`, `SigmoidLayer`, và `TanhLayer` với lan truyền ngược giải tích.

In [ ]:
class LinearLayer:
    def __init__(self, in_features, out_features):
        scale = np.sqrt(2.0 / (in_features + out_features))
        self.W = np.random.randn(in_features, out_features).astype(np.float32) * scale
        self.b = np.zeros((1, out_features), dtype=np.float32)
        self.dW = np.zeros_like(self.W)
        self.db = np.zeros_like(self.b)
        self.x = None

    def forward(self, x):
        self.x = x
        return np.dot(x, self.W) + self.b

    def backward(self, grad_output):
        self.dW = np.dot(self.x.T, grad_output)
        self.db = np.sum(grad_output, axis=0, keepdims=True)
        return np.dot(grad_output, self.W.T)


class LeakyReLULayer:
    def __init__(self, alpha=0.2):
        self.alpha = alpha
        self.x = None

    def forward(self, x):
        self.x = x
        return np.where(x > 0, x, self.alpha * x)

    def backward(self, grad_output):
        dx = np.where(self.x > 0, 1.0, self.alpha).astype(np.float32)
        return grad_output * dx


class SigmoidLayer:
    def __init__(self):
        self.out = None

    def forward(self, x):
        x_clipped = np.clip(x, -15.0, 15.0)
        self.out = 1.0 / (1.0 + np.exp(-x_clipped))
        return self.out

    def backward(self, grad_output):
        return grad_output * self.out * (1.0 - self.out)

print("[✓] Đã khởi tạo LinearLayer, LeakyReLULayer, và SigmoidLayer.")

## PHẦN 2: MẠNG PHÂN BIỆT (DISCRIMINATOR) & MẠNG SINH (GENERATOR)

- **Discriminator $D(x)$:** Nhận vào một mẫu $x$ và trả về xác suất mẫu đó là ảnh thật ($D(x) \in [0, 1]$).
- **Generator $G(z)$:** Nhận vào một vector nhiễu ngẫu nhiên $z \sim p_z$ và tạo ra mẫu dữ liệu giả $G(z)$.

In [ ]:
class DiscriminatorScratch:
    def __init__(self, input_dim=2, hidden_dim=32, lr=0.01):
        self.fc1 = LinearLayer(input_dim, hidden_dim)
        self.act1 = LeakyReLULayer(alpha=0.2)
        self.fc2 = LinearLayer(hidden_dim, 1)
        self.sig = SigmoidLayer()
        self.lr = lr
        self.layers = [self.fc1, self.fc2]

    def forward(self, x):
        h = self.act1.forward(self.fc1.forward(x))
        logits = self.fc2.forward(h)
        prob = self.sig.forward(logits)
        return prob, logits

    def backward(self, grad_logits):
        grad_h = self.act1.backward(self.fc2.backward(grad_logits))
        return self.fc1.backward(grad_h)

    def update_params(self):
        for layer in self.layers:
            layer.W -= self.lr * layer.dW
            layer.b -= self.lr * layer.db


class GeneratorScratch:
    def __init__(self, latent_dim=2, hidden_dim=32, output_dim=2, lr=0.01):
        self.fc1 = LinearLayer(latent_dim, hidden_dim)
        self.act1 = LeakyReLULayer(alpha=0.2)
        self.fc2 = LinearLayer(hidden_dim, output_dim)
        self.lr = lr
        self.layers = [self.fc1, self.fc2]

    def forward(self, z):
        h = self.act1.forward(self.fc1.forward(z))
        return self.fc2.forward(h)

    def backward(self, grad_output):
        grad_h = self.act1.backward(self.fc2.backward(grad_output))
        return self.fc1.backward(grad_h)

    def update_params(self):
        for layer in self.layers:
            layer.W -= self.lr * layer.dW
            layer.b -= self.lr * layer.db

print("[✓] Đã khởi tạo DiscriminatorScratch và GeneratorScratch.")

## PHẦN 3: HÀM MẤT MÁT VÀ GIẢI PHÁP NON-SATURATING LOSS

- **Minimax gốc:** $\min_G \log(1 - D(G(z)))$. Khi $D$ mạnh, gradient tiến về $0$ làm $G$ bị tê liệt.
- **Non-Saturating Loss (Goodfellow 2014):** $\max_G \log D(G(z)) \iff \min_G -\log D(G(z))$. Cung cấp gradient mạnh mẽ ngay từ epoch đầu tiên!

In [ ]:
from gan_scratch import GANLosses

# Kiểm thử nhanh hàm mất mát GANLosses
d_real_dummy = np.array([[0.9], [0.85]], dtype=np.float32)
d_fake_dummy = np.array([[0.1], [0.15]], dtype=np.float32)
loss_d, g_real, g_fake = GANLosses.discriminator_loss_and_grad(d_real_dummy, d_fake_dummy)
loss_g, g_logits = GANLosses.generator_loss_and_grad(d_fake_dummy, use_non_saturating=True)
print(f"[*] Kiểm thử loss Discriminator: {loss_d:.4f}")
print(f"[*] Kiểm thử loss Generator (Non-saturating): {loss_g:.4f}")

## PHẦN 4: ĐỐI CHIẾU GIÁ TRỊ HỌC THUẬT (ACADEMIC VALUE BENCHMARK)

So sánh khả năng học phân phối đa mode (Bimodal distribution):
- **Mô hình MLE Tiền-GAN:** Lấy trung bình cộng hai mode, làm rơi mẫu vào khoảng trống ở giữa (Mode-averaging).
- **Mô hình GAN (Goodfellow 2014):** Loại bỏ hoàn toàn mẫu rơi vào khoảng trống, mẫu sinh bám sát chính xác hai mode thật.

In [ ]:
from gan_scratch import run_academic_value_comparison

# Chạy kiểm thử đối chuẩn học thuật trực tiếp
run_academic_value_comparison()

## 🎓 TỔNG KẾT & BÀI HỌC LỊCH SỬ TỪ BÀI BÁO GAN (NEURIPS 2014)

1. **Mô Hình Sinh Ngầm Định (Implicit Generative Model):** Loại bỏ sự phụ thuộc vào việc tính toán tích phân hàm mật độ $p(x)$, giải phóng mô hình sinh trên không gian dữ liệu nhiều chiều.
2. **Lý Thuyết Trò Chơi Trong Học Sâu:** Thiết lập sự hội tụ về điểm cân bằng Nash thông qua bài toán đối kháng Minimax.
3. **Độ Sắc Nét Vượt Trội:** Tránh bẫy bình quân hóa các mode (Mode-Averaging) của Maximum Likelihood, giúp hình ảnh sinh ra đạt độ tương phản và biên cạnh cực kỳ sắc nét.
4. **Bệ phóng cho Siêu Mô Hình Tạo Sinh:** Mở đường trực tiếp cho DCGAN, WGAN, CycleGAN, Pix2Pix và dòng họ StyleGAN thống trị ngành sáng tạo nội dung thị giác.